Check the model names from Pytorch Computer Vision (Timm) library:

In [1]:
# Hyperparameters
MODEL = "seresnext101_32x4d.gluon_in1k_timm"
DEVICE_ID = 4
BATCH_SIZE = 128
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
LOG_DIR = f"./{MODEL}_new_dataset_logs/"

In [2]:
# Pytorch vision package (Timm)
import timm

# List relevant models
timm_name = MODEL[:-5] # Removing _timm in the name
convnext_models = timm.list_models(f'{timm_name}*')
print(convnext_models)

pretrained_models = timm.list_models(f'{timm_name}*', pretrained=True)
print(pretrained_models)

[]
['seresnext101_32x4d.gluon_in1k']


In [3]:
# Pytorch packages
import torch
from torch import nn
# from torch.nn import functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Other packages
import os
import json
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

In [4]:
# Load the train and validation dataset
train_datapath = os.path.join(DATA_DIR, 'train_aug_labels.csv')
val_datapath = os.path.join(DATA_DIR, 'val_labels.csv')
test_datapath = os.path.join(DATA_DIR, 'test_labels.csv')

train_df = pd.read_csv(train_datapath) 
val_df = pd.read_csv(val_datapath) 
test_df = pd.read_csv(test_datapath)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["species"])
val_df["label"] = le.transform(val_df["species"])
test_df["label"] = le.transform(test_df["species"])
num_classes = len(le.classes_)
print(f"Total images, Train: {len(train_df['label'])}, Validation: {len(val_df['label'])}, Test: {len(test_df['label'])}")
print(f"Total classes: {num_classes}")

Total images, Train: 34722, Validation: 1158, Test: 771
Total classes: 160


In [5]:
train_path = os.path.join(DATA_DIR, "train", 'aug_images') # Path for the training data
val_path = os.path.join(DATA_DIR, "valid", 'images') # Path for validation data
test_path = os.path.join(DATA_DIR, "test", 'images') # Path for testing data

In [6]:
missing = 0
for i in range(len(train_df)):
    p = os.path.join(train_path, train_df.loc[i,"images"])
    if not os.path.exists(p):
        print("Missing:", p)
        missing += 1
print("Missing training images:", missing)

missing = 0
for i in range(len(val_df)):
    p = os.path.join(val_path, val_df.loc[i,"images"])
    if not os.path.exists(p):
        print("Missing:", p)
        missing += 1
print("Missing val images:", missing)

missing = 0
for i in range(len(test_df)):
    p = os.path.join(test_path, test_df.loc[i,"images"])
    if not os.path.exists(p):
        print("Missing:", p)
        missing += 1
print("Missing test images:", missing)

Missing training images: 0
Missing val images: 0
Missing test images: 0


In [7]:
class SpeciesDataset(Dataset):
    def __init__(self, df, img_dir, image_size):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.4925, 0.4475, 0.3490), # Custom normalization values for beemachine_small_2025_v3 (segmentation) dataset
                             std=(0.2392, 0.2265, 0.2213))
        # transforms.Normalize(mean=[0.485, 0.456, 0.406], # Imagenet Normalization values
        #              std=[0.229, 0.224, 0.225])
    ])
        self.num_classes = self.df["label"].nunique()
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, "images"]
        label = self.df.loc[idx, "label"]

        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [8]:
train_dataset = SpeciesDataset(df=train_df, img_dir=train_path, image_size=IMAGE_SIZE)
val_dataset = SpeciesDataset(df=val_df, img_dir=val_path, image_size=IMAGE_SIZE)
test_dataset = SpeciesDataset(df=test_df, img_dir=test_path, image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

num_classes = train_dataset.num_classes
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 160 | Train: 34722 | Val: 1158 | Test: 771


In [9]:
# Get one batch from the train loader
images, labels = next(iter(train_loader))

print("Batch image tensor shape:", images.shape)
print("Batch label tensor shape:", labels.shape)

Batch image tensor shape: torch.Size([128, 3, 224, 224])
Batch label tensor shape: torch.Size([128])


In [10]:
classes = le.classes_
len(classes), classes

(160,
 array(['Bombus_affinis', 'Bombus_alpinus', 'Bombus_appositus',
        'Bombus_ardens', 'Bombus_argillaceus', 'Bombus_armeniacus',
        'Bombus_ashtoni', 'Bombus_atripes', 'Bombus_auricomus',
        'Bombus_baeri', 'Bombus_balteatus', 'Bombus_barbutellus',
        'Bombus_bellicosus', 'Bombus_bicoloratus', 'Bombus_bifarius',
        'Bombus_bimaculatus', 'Bombus_bohemicus', 'Bombus_borealis',
        'Bombus_brachycephalus', 'Bombus_brasiliensis', 'Bombus_breviceps',
        'Bombus_butteli', 'Bombus_californicus', 'Bombus_caliginosus',
        'Bombus_campestris', 'Bombus_centralis', 'Bombus_citrinus',
        'Bombus_coccineus', 'Bombus_confusus', 'Bombus_consobrinus',
        'Bombus_coreanus', 'Bombus_crotchii', 'Bombus_cryptarum',
        'Bombus_cullumanus', 'Bombus_dahlbomii', 'Bombus_diligens',
        'Bombus_distinguendus', 'Bombus_diversus', 'Bombus_excellens',
        'Bombus_eximius', 'Bombus_fervidus', 'Bombus_festivus',
        'Bombus_flavescens', 'Bombus_fla

In [11]:
# Loading the model
device = torch.device(f"cuda:{DEVICE_ID}")
num_classes = len(classes)
model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes)
model.to(device)

# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss()  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [12]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0

    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [13]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


[Train]:   0%|          | 0/272 [00:00<?, ?it/s]

 Epoch 1/100 | Train Loss: 1.8803 | Train Acc: 0.5889 | Val Loss: 1.3083 | Val Acc: 0.6718


 Epoch 2/100 | Train Loss: 0.1335 | Train Acc: 0.9775 | Val Loss: 1.2489 | Val Acc: 0.6813


 Epoch 3/100 | Train Loss: 0.0174 | Train Acc: 0.9984 | Val Loss: 1.2405 | Val Acc: 0.6943


...

 Epoch 96/100 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 1.3325 | Val Acc: 0.7185


 Epoch 97/100 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 1.3283 | Val Acc: 0.7150


 Epoch 98/100 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 1.3186 | Val Acc: 0.7219


 Epoch 99/100 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 1.3203 | Val Acc: 0.7211


 Epoch 100/100 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 1.3220 | Val Acc: 0.7193
Training complete!
CPU times: user 4h 46min 9s, sys: 1h 14min 58s, total: 6h 1min 8s
Wall time: 6h 3min 17s


In [14]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.6209 | Test Acc: 0.6693
